# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

# A2 — Knowledge-Base Demo

This notebook demonstrates the constructed knowledge base through:

1. **OCR quality** on a small sample of indexed chunks.
2. **Index statistics**:
   - number of chunks indexed
   - embedding dimension
   - FAISS index type
   - corpus coverage: pages and words indexed vs. total
3. **One working retrieval example**:
   - a real user query
   - top retrieved chunk
   - source book/page
   - whether the retrieved page contains the relevant answer

In [1]:
# IMPLEMENT: run OCR quality + one retrieval, end to end
from pathlib import Path
import json
import re
import random

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

INDEX_DIR = Path("../data/interim/index")

FAISS_PATH = INDEX_DIR / "index.faiss"
CHUNKS_PATH = INDEX_DIR / "chunks.jsonl"

assert FAISS_PATH.exists(), f"Missing FAISS index: {FAISS_PATH}"
assert CHUNKS_PATH.exists(), f"Missing chunk metadata: {CHUNKS_PATH}"

print(f"Index:  {FAISS_PATH}")
print(f"Chunks: {CHUNKS_PATH}")

/Users/mesbah/Desktop/Agri-Bot/CSE429-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index:  ../data/interim/index/index.faiss
Chunks: ../data/interim/index/chunks.jsonl


In [2]:
# Load FAISS index
index = faiss.read_index(str(FAISS_PATH))

# Load chunk metadata
chunks = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks):,} chunk records")
print(f"FAISS contains {index.ntotal:,} vectors")

assert index.ntotal == len(chunks), (
    f"Mismatch: FAISS has {index.ntotal} vectors but "
    f"chunks.jsonl has {len(chunks)} records"
)

Loaded 1,814 chunk records
FAISS contains 1,814 vectors


In [3]:
# ------------------------------------------------------------
# Basic index statistics
# ------------------------------------------------------------

n_chunks = index.ntotal
embedding_dim = index.d

index_type = type(index).__name__

print("=== Index Statistics ===")
print(f"Chunks indexed       : {n_chunks:,}")
print(f"Embedding dimension  : {embedding_dim}")
print(f"Index type            : {index_type}")
print(f"Index metric          : Inner Product")

=== Index Statistics ===
Chunks indexed       : 1,814
Embedding dimension  : 1024
Index type            : IndexFlatIP
Index metric          : Inner Product


In [4]:
# ------------------------------------------------------------
# Pages represented in the index
# ------------------------------------------------------------

indexed_page_ids = set()

for chunk in chunks:
    indexed_page_ids.update(chunk.get("page_ids", []))

indexed_pages = len(indexed_page_ids)


# ------------------------------------------------------------
# Words represented in the index
# ------------------------------------------------------------

def count_words(text):
    """
    Unicode-friendly approximate word count.

    For Bengali/English OCR text, whitespace-separated tokens are
    a useful corpus-level measure.
    """
    return len(text.split())


indexed_words = sum(
    count_words(chunk.get("text", ""))
    for chunk in chunks
)

print(f"Pages represented in index : {indexed_pages:,}")
print(f"Words represented in index : {indexed_words:,}")

Pages represented in index : 742
Words represented in index : 129,496


In [5]:
# ------------------------------------------------------------
# Total corpus from OCR region caches
# ------------------------------------------------------------

OCR_DIR = Path("../data/interim/ocr")

assert OCR_DIR.exists(), f"OCR directory not found: {OCR_DIR}"


def load_ocr_pages(ocr_dir):
    """
    Return:
        total_pages
        total_words
        page_ids
    based on OCR region caches.
    """

    page_ids = set()
    total_words = 0

    json_files = sorted(ocr_dir.glob("*/*.json"))

    for path in json_files:
        doc_id = path.parent.name
        page_stem = path.stem

        page_id = f"{doc_id}:{page_stem}"
        page_ids.add(page_id)

        try:
            with open(path, "r", encoding="utf-8") as f:
                regions = json.load(f)
        except Exception:
            continue

        for region in regions:
            text = region.get("corrected_text", "")
            total_words += count_words(text)

    return len(page_ids), total_words, page_ids


total_pages, total_words, all_page_ids = load_ocr_pages(OCR_DIR)

print(f"Total OCR pages  : {total_pages:,}")
print(f"Total OCR words  : {total_words:,}")

Total OCR pages  : 742
Total OCR words  : 131,123


In [6]:
page_coverage = (
    100 * indexed_pages / total_pages
    if total_pages else 0
)

word_coverage = (
    100 * indexed_words / total_words
    if total_words else 0
)

coverage_df = pd.DataFrame([
    {
        "Metric": "Pages",
        "Indexed": indexed_pages,
        "Total": total_pages,
        "Coverage (%)": page_coverage,
    },
    {
        "Metric": "Words",
        "Indexed": indexed_words,
        "Total": total_words,
        "Coverage (%)": word_coverage,
    },
])

coverage_df["Coverage (%)"] = coverage_df["Coverage (%)"].round(2)

display(coverage_df)

,Metric,Indexed,Total,Coverage (%)
0,Pages,742,742,100.00
1,Words,129496,131123,98.76


In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import jiwer
from IPython.display import display


# ============================================================
# Configuration
# ============================================================

INDEX_DIR = Path("../data/interim/index")
CHUNKS_PATH = INDEX_DIR / "chunks.jsonl"
GT_ROOT = Path("../data/ground-truth-ocr")

N_EXAMPLES = 4


# ============================================================
# Load all indexed chunks
# ============================================================

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    all_chunks = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print(f"Loaded {len(all_chunks):,} chunks from {CHUNKS_PATH}")


# ============================================================
# Normalize text
# ============================================================

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# ============================================================
# Find GT for a page
# ============================================================

def find_gt(book: str, page_stem: str):
    """
    Locate the page-level GT file.

    Expected examples:

        ground-truth-ocr/
            BOOK/
                page-001.txt
                page-002.txt
    """

    match = re.search(r"(\d+)$", page_stem)

    if not match:
        return None

    page_num = int(match.group(1))
    book_dir = GT_ROOT / book

    if not book_dir.exists():
        return None

    candidates = [
        book_dir / f"page-{page_num:03d}.txt",
        book_dir / f"page-{page_num}.txt",
        book_dir / f"{page_num:03d}.txt",
        book_dir / f"{page_num}.txt",
    ]

    for path in candidates:
        if path.exists():
            return path
    return None


# ============================================================
# Group ALL chunks by page
# ============================================================

page_chunks = {}

for chunk in all_chunks:

    page_ids = chunk.get("page_ids", [])

    for page_id in page_ids:
        page_chunks.setdefault(page_id, []).append(chunk)


print(f"Pages represented in index: {len(page_chunks):,}")


# ============================================================
# Sort chunks within each page
#
# Chunk IDs look like:
#
#   BOOK:page-010:0-0
#   BOOK:page-010:1-0
#   BOOK:page-010:2-0
#
# The final part is group/piece ordering.
# ============================================================

def chunk_sort_key(chunk):
    chunk_id = chunk["id"]

    match = re.search(r":(\d+)-(\d+)$", chunk_id)

    if match:
        return (
            int(match.group(1)),
            int(match.group(2)),
        )

    return (10**9, 10**9)


for page_id in page_chunks:
    page_chunks[page_id].sort(key=chunk_sort_key)


# ============================================================
# Select pages that have GT
# ============================================================

evaluatable_pages = []

for page_id, chunks in page_chunks.items():

    if ":" not in page_id:
        continue

    book, page_stem = page_id.split(":", 1)

    gt_path = find_gt(book, page_stem)

    if gt_path is not None:
        evaluatable_pages.append(
            (page_id, gt_path)
        )


print(
    f"Pages with both indexed chunks and GT: "
    f"{len(evaluatable_pages):,}"
)


# ============================================================
# Pick examples
#
# Here we simply take the first 4 evaluatable pages.
# ============================================================

examples = evaluatable_pages[:N_EXAMPLES]


# ============================================================
# Evaluate each COMPLETE PAGE
# ============================================================

records = []

print("\n" + "=" * 100)
print("PAGE-LEVEL OCR EVALUATION")
print("=" * 100)

for i, (page_id, gt_path) in enumerate(examples, start=1):

    chunks = page_chunks[page_id]

    # --------------------------------------------------------
    # Combine ALL chunks belonging to this page
    # --------------------------------------------------------

    combined_ocr = "\n".join(
        chunk["text"]
        for chunk in chunks
        if chunk.get("text", "").strip()
    )

    # --------------------------------------------------------
    # Load COMPLETE page GT
    # --------------------------------------------------------

    gt_text = gt_path.read_text(encoding="utf-8")

    # Normalize only for scoring
    ocr_norm = normalize_text(combined_ocr)
    gt_norm = normalize_text(gt_text)

    # --------------------------------------------------------
    # Compute page-level CER / WER
    # --------------------------------------------------------

    cer = jiwer.cer(gt_norm, ocr_norm) if ocr_norm else 1.0
    wer = jiwer.wer(gt_norm, ocr_norm) if ocr_norm else 1.0

    records.append({
        "Example": i,
        "Page": page_id,
        "Chunks": len(chunks),
        "OCR chars": len(ocr_norm),
        "GT chars": len(gt_norm),
        "CER": cer,
        "WER": wer,
        "GT file": str(gt_path),
    })

    # --------------------------------------------------------
    # Display page
    # --------------------------------------------------------

    print("\n" + "-" * 100)
    print(f"Example {i}")
    print(f"Page   : {page_id}")
    print(f"Chunks : {len(chunks)}")
    print(f"CER    : {cer:.4f} ({cer * 100:.2f}%)")
    print(f"WER    : {wer:.4f} ({wer * 100:.2f}%)")
    print("-" * 100)

    print("COMBINED OCR:")
    print(combined_ocr)

    print("\nGROUND TRUTH:")
    print(gt_text)


# ============================================================
# Summary table
# ============================================================

print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)

results_df = pd.DataFrame(records)

if not results_df.empty:

    display(
        results_df[
            [
                "Example",
                "Page",
                "Chunks",
                "OCR chars",
                "GT chars",
                "CER",
                "WER",
            ]
        ].style.format({
            "CER": "{:.4f}",
            "WER": "{:.4f}",
        })
    )

    print(
        f"\nMean CER: "
        f"{results_df['CER'].mean():.4f} "
        f"({results_df['CER'].mean() * 100:.2f}%)"
    )

    print(
        f"Mean WER: "
        f"{results_df['WER'].mean():.4f} "
        f"({results_df['WER'].mean() * 100:.2f}%)"
    )

else:
    print("No pages with both indexed chunks and GT were found.")

Loaded 1,814 chunks from ../data/interim/index/chunks.jsonl
Pages represented in index: 742
Pages with both indexed chunks and GT: 742

PAGE-LEVEL OCR EVALUATION

----------------------------------------------------------------------------------------------------
Example 1
Page   : Bharater-Krishi-Babyasthar-Parichay:page-004
Chunks : 1
CER    : 0.4200 (42.00%)
WER    : 0.6154 (61.54%)
----------------------------------------------------------------------------------------------------
COMBINED OCR:
পথম খণ্ড

আভূমি পাবলিশিং কোম্পানী
৭৯, মহাজস। গাক্ষী পোড, কলিকাতা-৯

GROUND TRUTH:
ভারতের কৃষি-ব্যবস্থার পরিচয়



প্রথম খণ্ড



শ্রীভূমি পাবলিশিং কোম্পানী
৭৯, মহাত্মা গান্ধী রোড, কলিকাতা-৯

----------------------------------------------------------------------------------------------------
Example 2
Page   : Bharater-Krishi-Babyasthar-Parichay:page-008
Chunks : 1
CER    : 0.2083 (20.83%)
WER    : 0.5306 (53.06%)
---------------------------------------------------------------------------------

,Example,Page,Chunks,OCR chars,GT chars,CER,WER
0,1,Bharater-Krishi-Babyasthar-Parichay:page-004,1,65,100,0.4200,0.6154
1,2,Bharater-Krishi-Babyasthar-Parichay:page-008,1,605,648,0.2083,0.5306
2,3,Bharater-Krishi-Babyasthar-Parichay:page-009,1,518,581,0.2857,0.5938
3,4,Bharater-Krishi-Babyasthar-Parichay:page-010,3,1333,1353,0.0473,0.1143



Mean CER: 0.2403 (24.03%)
Mean WER: 0.4635 (46.35%)


In [9]:
# ------------------------------------------------------------
# Embedding model used by the KB
# ------------------------------------------------------------

MODEL_NAME = "BAAI/bge-m3"

embedder = SentenceTransformer(MODEL_NAME)

print(f"Loaded embedding model: {MODEL_NAME}")
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

assert (
    embedder.get_sentence_embedding_dimension()
    == index.d
), "Embedding dimension does not match FAISS index!"

Loaded embedding model: BAAI/bge-m3
Embedding dimension: 1024


In [10]:
def retrieve(query: str, k: int = 5):
    """
    Retrieve top-k chunks using the same embedding configuration
    used during index construction.
    """

    query_vector = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    scores, indices = index.search(query_vector, k)

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        chunk = chunks[int(idx)].copy()

        results.append({
            "rank": rank,
            "index_row": int(idx),
            "score": float(score),
            "chunk": chunk,
        })

    return results

In [14]:
# ------------------------------------------------------------
# One verified working retrieval example
# ------------------------------------------------------------

QUERY = "কোন কোন বৈশিষ্ট্যের দ্বারা মৃত্তিকার উর্বরতা ও অনুর্বরতা নির্দেশ করা যায় এবং কোন মৃত্তিকা কোন ফসলের জন্য উপযোগী?"

results = retrieve(QUERY, k=5)

print(f"Query: {QUERY}")
print()

Query: কোন কোন বৈশিষ্ট্যের দ্বারা মৃত্তিকার উর্বরতা ও অনুর্বরতা নির্দেশ করা যায় এবং কোন মৃত্তিকা কোন ফসলের জন্য উপযোগী?



In [15]:
top = results[0]
top_chunk = top["chunk"]

print("=" * 90)
print("TOP RETRIEVED CHUNK")
print("=" * 90)

print(f"Rank       : {top['rank']}")
print(f"FAISS row  : {top['index_row']}")
print(f"Score      : {top['score']:.4f}")
print(f"Chunk ID   : {top_chunk['id']}")
print(f"Book       : {top_chunk['doc_id']}")
print(f"Page       : {top_chunk['page_ids']}")
print()
print(top_chunk["text"])

TOP RETRIEVED CHUNK
Rank       : 1
FAISS row  : 1120
Score      : 0.7303
Chunk ID   : Krishi-Bigyan:page-206:1-0
Book       : Krishi-Bigyan
Page       : ['Krishi-Bigyan:page-206']

মৃত্তিকার বর্ণ এবং যাত্ররক অবস্থার প্রতি লক্ষ্য করিয়াও উহার
উর্বরতা এবং অনুর্ব্বরা নির্দেশ করা যায়। কালো এবঙ গীত বর্ণের
মৃত্তিকা সাধারণতঃ উর্ব্বরা হইয়া থাকে; এবং সাদা, ধূমর ও অধিক লাল
বর্ণের মৃত্তিকা সাধারণতঃ অঙ্কুর হইয়া থাকে। যে মৃৎকার শীতকালে,
অর্থাৎ নিতান্ত শুষ্ক দিনেও, লাঙ্গল দ্বারা অনায়াসে কর্ষণ করা যায় এইরূপ
হাল্কা মৃত্তিকা স্বভাবতঃই উর্বরা। বৃষ্টিপাত না হইলে যে মৃৎকার
সহজে কর্ষণ করা যায় না এইরূপ দৃঢ় মৃত্তিকা অধিকাংশ স্থলেই উর্বরা হয়
না। বৃষ্টিবারি-পতন মাত্রই যে জমি হইতে নিঃসৃত হইয়া যায় এবং যে
জমিতে বৃষ্টিবারি অধিককাল দাঁড়াইয়া থাকে, এই উভয় প্রকার মৃত্তিকাই
উর্বর হইলেও কৃষিকার্য্যের পক্ষে উপযোগী নহে। যে মৃত্তিকা বৃষ্টিবারি
দ্বারা বিগলিত ও বিধৌত হইয়া স্থানান্তরে চলিয়া যায় তাহাও কৃষিকার্য্যের
উপযুক্ত নহে। কঠিন এবং হাল্কা—এই উভয় প্রকার মৃত্তিকাতে আপন
আপন স্বভাবের উপযোগী ফসল জন্মিতে পারে; 

In [20]:
print(f"Showing {min(4, len(results))} retrieved chunks:\n")

for r in results[:4]:
    chunk = r["chunk"]

    print("=" * 100)
    print(f"Rank  : {r['rank']}")
    print(f"Score : {r['score']:.4f}")
    print(f"Book  : {chunk['doc_id']}")
    print(f"Page  : {', '.join(chunk.get('page_ids', []))}")
    print("-" * 100)
    print(chunk["text"])
    print()

Showing 4 retrieved chunks:

Rank  : 1
Score : 0.7303
Book  : Krishi-Bigyan
Page  : Krishi-Bigyan:page-206
----------------------------------------------------------------------------------------------------
মৃত্তিকার বর্ণ এবং যাত্ররক অবস্থার প্রতি লক্ষ্য করিয়াও উহার
উর্বরতা এবং অনুর্ব্বরা নির্দেশ করা যায়। কালো এবঙ গীত বর্ণের
মৃত্তিকা সাধারণতঃ উর্ব্বরা হইয়া থাকে; এবং সাদা, ধূমর ও অধিক লাল
বর্ণের মৃত্তিকা সাধারণতঃ অঙ্কুর হইয়া থাকে। যে মৃৎকার শীতকালে,
অর্থাৎ নিতান্ত শুষ্ক দিনেও, লাঙ্গল দ্বারা অনায়াসে কর্ষণ করা যায় এইরূপ
হাল্কা মৃত্তিকা স্বভাবতঃই উর্বরা। বৃষ্টিপাত না হইলে যে মৃৎকার
সহজে কর্ষণ করা যায় না এইরূপ দৃঢ় মৃত্তিকা অধিকাংশ স্থলেই উর্বরা হয়
না। বৃষ্টিবারি-পতন মাত্রই যে জমি হইতে নিঃসৃত হইয়া যায় এবং যে
জমিতে বৃষ্টিবারি অধিককাল দাঁড়াইয়া থাকে, এই উভয় প্রকার মৃত্তিকাই
উর্বর হইলেও কৃষিকার্য্যের পক্ষে উপযোগী নহে। যে মৃত্তিকা বৃষ্টিবারি
দ্বারা বিগলিত ও বিধৌত হইয়া স্থানান্তরে চলিয়া যায় তাহাও কৃষিকার্য্যের
উপযুক্ত নহে। কঠিন এবং হাল্কা—এই উভয় প্রকার মৃত্তিকাতে আপন
আপন স্বভাবে